# Surface Code: Noisy Simulation

This notebook demonstrates:
1. Noiseless surface code memory experiments
2. Noisy simulations with depolarizing noise
3. Logical error rate measurement
4. Distance scaling behavior

**Note**: Proper MWPM decoding with circuit-level noise requires a space-time matching graph.
This notebook shows raw logical error rates from the code structure.

In [ ]:
import numpy as np

from pecos.compilation_pipeline import compile_guppy_to_hugr
from pecos.guppy.surface import get_num_qubits, make_surface_code
from pecos.qec.surface import SurfacePatch, plot_surface_code
from selene_sim import DepolarizingErrorModel, IdealErrorModel, SimpleRuntime, Stim, build

## Configuration

In [ ]:
NUM_ROUNDS = 1  # Syndrome extraction rounds
NUM_SHOTS = 500  # Shots per experiment
BASIS = "Z"  # Memory experiment basis

## Helper Functions

In [ ]:
def get_logical_qubits(distance: int, basis: str) -> tuple:
    """Get qubits in the logical operator."""
    patch = SurfacePatch.create(distance=distance)
    if basis == "Z":
        return patch.geometry.logical_z.data_qubits
    else:
        return patch.geometry.logical_x.data_qubits


def run_memory_experiment(
    distance: int,
    num_rounds: int,
    num_shots: int,
    basis: str,
    error_model,
) -> dict:
    """Run memory experiment and compute logical error rate.
    
    For Z-basis: prepare |0_L>, measure in Z basis, check logical Z parity.
    For X-basis: prepare |+_L>, measure in X basis, check logical X parity.
    """
    logical_qubits = get_logical_qubits(distance, basis)
    
    # Build circuit
    num_qubits = get_num_qubits(distance)
    prog = make_surface_code(distance=distance, num_rounds=num_rounds, basis=basis)
    hugr_bytes = compile_guppy_to_hugr(prog)
    instance = build(hugr_bytes, name=f"surface_d{distance}")
    
    # Run
    num_logical_errors = 0
    
    for shot_results in instance.run_shots(
        simulator=Stim(),
        n_qubits=num_qubits,
        n_shots=num_shots,
        error_model=error_model,
        runtime=SimpleRuntime(),
        n_processes=1,
    ):
        results = {name: list(values) for name, values in shot_results}
        final = results.get("final", [])
        
        if final:
            parity = sum(final[q] for q in logical_qubits) % 2
            if parity != 0:
                num_logical_errors += 1
    
    return {
        "distance": distance,
        "num_shots": num_shots,
        "num_logical_errors": num_logical_errors,
        "logical_error_rate": num_logical_errors / num_shots,
    }

## Surface Code Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, d in enumerate([3, 5, 7]):
    fig_d, ax_d = plot_surface_code(d, show_cnot_order=False)
    ax_d.set_title(f"Distance {d}", fontsize=12)
    fig_d.show()

## Part 1: Noiseless Verification

Verify the circuit works correctly with no noise.

In [ ]:
print("=== Noiseless Verification ===")
print()

for d in [3, 5, 7]:
    result = run_memory_experiment(
        distance=d,
        num_rounds=NUM_ROUNDS,
        num_shots=100,
        basis=BASIS,
        error_model=IdealErrorModel(),
    )
    print(f"d={d}: {result['num_logical_errors']}/{result['num_shots']} errors "
          f"(LER = {result['logical_error_rate']:.4f})")

## Part 2: Noisy Simulation

Add depolarizing noise and measure logical error rates.
The depolarizing model applies:
- Single-qubit errors after 1Q gates
- Two-qubit errors after 2Q gates  
- Measurement errors
- Initialization errors

In [ ]:
distances = [3, 5, 7]
error_rates = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1]

results = []

print("=== Noisy Simulation ===")
for d in distances:
    print(f"\nDistance {d}:")
    for p in error_rates:
        error_model = DepolarizingErrorModel(
            p_1q=p, p_2q=p, p_meas=p, p_init=p
        )
        
        result = run_memory_experiment(
            distance=d,
            num_rounds=NUM_ROUNDS,
            num_shots=NUM_SHOTS,
            basis=BASIS,
            error_model=error_model,
        )
        
        results.append((d, p, result['logical_error_rate']))
        print(f"  p={p:.3f}: LER={result['logical_error_rate']:.3f} "
              f"({result['num_logical_errors']}/{NUM_SHOTS})")

## Part 3: Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 7))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
markers = ['o', 's', '^']

for i, d in enumerate(distances):
    d_results = [(p, ler) for (dd, p, ler) in results if dd == d]
    ps, lers = zip(*d_results)
    
    # Replace zeros with small value for log scale
    lers = [max(ler, 1e-4) for ler in lers]
    
    ax.plot(ps, lers, f'{markers[i]}-', color=colors[i], 
            label=f'd={d}', markersize=10, linewidth=2)

# Reference lines
ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, label='Random (50%)')
ax.axvline(x=0.01, color='red', linestyle='--', alpha=0.3, label='~1% threshold')

ax.set_xlabel('Physical Error Rate', fontsize=14)
ax.set_ylabel('Logical Error Rate', fontsize=14)
ax.set_title('Surface Code Logical Error Rate vs Physical Error Rate\n(Single Round, No Decoding)', fontsize=14)
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend(fontsize=12, loc='lower right')
ax.grid(True, alpha=0.3, which='both')
ax.set_xlim(0.0008, 0.15)
ax.set_ylim(1e-4, 1.0)

plt.tight_layout()
plt.show()

## Part 4: Analysis

**Important**: Without decoding, larger distance gives *higher* logical error rates!
This is because more qubits = more chances for errors to corrupt the logical state.

The surface code's error suppression comes from the **decoder**, not just the code structure.
With proper MWPM decoding:
- Below threshold: larger distance gives lower logical error rate
- Above threshold: larger distance doesn't help
- Threshold (~1%): curves cross

In [ ]:
# Compare distances at specific error rates
print("=== Distance Comparison ===")
print()

for p in [0.001, 0.005, 0.01, 0.02]:
    print(f"p = {p}:")
    for d in distances:
        ler = next((ler for (dd, pp, ler) in results if dd == d and pp == p), None)
        if ler is not None:
            print(f"  d={d}: LER = {ler:.4f}")
    print()

## Note on Decoding

This notebook shows **raw** logical error rates without MWPM decoding.
The decoder infrastructure exists in `pecos_rslib.decoders`:

```python
from pecos_rslib.decoders import CheckMatrix, PyMatchingDecoder

# Create decoder from parity check matrix
Hx = patch.get_parity_matrix('X')
decoder = PyMatchingDecoder.from_check_matrix(
    CheckMatrix.from_dense(Hx.tolist())
)

# Decode syndrome
result = decoder.decode(syndrome)
correction = result.correction
```

For **circuit-level noise** (with measurement errors), proper decoding requires:
1. A space-time matching graph that correlates syndromes across rounds
2. Detection events (syndrome changes) rather than raw syndromes
3. Proper handling of the boundary conditions

This is an area for future enhancement.

## Summary

This notebook demonstrated:

1. **Surface code circuits**: Generated using `make_surface_code()` from `pecos.guppy.surface`
2. **Noisy simulation**: Using `DepolarizingErrorModel` from `selene_sim`
3. **Logical error measurement**: Checking parity of logical operator qubits
4. **Raw error behavior**: Without decoding, larger distance = more errors

**Key insight**: The surface code threshold (~1%) only manifests with proper decoding.
Without a decoder, larger codes have more qubits that can be corrupted, leading to higher
logical error rates. The decoder is essential for error suppression!

**Next steps**:
- Implement space-time matching for circuit-level decoding
- Use detection events (syndrome changes between rounds)
- Compare decoded vs raw error rates to show threshold behavior